In [27]:
#TASK - 01:)

import math
import os
import copy
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm


os.makedirs("models", exist_ok=True)



DEVICE = torch.device(

    "cuda" if torch.cuda.is_available()

    else "cpu"
)

print("Using Device:", DEVICE)


BATCH_SIZE = 32

EMB_SIZE = 128

NHEAD = 4

FFN_HID_DIM = 256

NUM_ENCODER_LAYERS = 3

NUM_DECODER_LAYERS = 3

DROPOUT = 0.1

LR = 0.0001

EPOCHS = 25

MAX_SAMPLES = 15000

SRC_VOCAB_SIZE = 8000

TGT_VOCAB_SIZE = 8000

MIN_DELTA = 0.001

PATIENCE=3

Using Device: cuda


In [28]:

#training


TRAIN_SRC = "/kaggle/input/datasets/rishav2307/cerecin/opus.en-hi-train.en"

TRAIN_TRG = "/kaggle/input/datasets/rishav2307/cerecin/opus.en-hi-train.hi"

#testing

TEST_SRC = "/kaggle/input/datasets/rishav2307/cerecin/opus.en-hi-test.en"

TEST_TRG = "/kaggle/input/datasets/rishav2307/cerecin/opus.en-hi-test.hi"

#tokenisatioon

def tokenize_en(text):

    return text.lower().strip().split()

def tokenize_hi(text):

    return text.strip().split()

In [29]:
#datasets

class TranslationDataset(Dataset):

    def __init__(

        self,

        src_path,

        trg_path,

        max_samples=None
    ):

        with open(

            src_path,

            encoding="utf-8"
        ) as f:

            self.src_sentences = f.readlines()

        with open(

            trg_path,

            encoding="utf-8"
        ) as f:

            self.trg_sentences = f.readlines()

        # ==================================
        # OPTIONAL SAMPLE LIMIT
        # ==================================

        if max_samples is not None:

            self.src_sentences = self.src_sentences[:max_samples]

            self.trg_sentences = self.trg_sentences[:max_samples]

        print(

            f"✅ Loaded {len(self.src_sentences)} sentence pairs"
        )

    def __len__(self):

        return len(self.src_sentences)

    def __getitem__(self, idx):

        src = tokenize_en(

            self.src_sentences[idx]
        )

        trg = tokenize_hi(

            self.trg_sentences[idx]
        )

        return src, trg


train_dataset = TranslationDataset(

    TRAIN_SRC,

    TRAIN_TRG,

    max_samples=60000
)

test_dataset = TranslationDataset(

    TEST_SRC,

    TEST_TRG
)

✅ Loaded 60000 sentence pairs
✅ Loaded 2000 sentence pairs


In [30]:

special_tokens = [

    '<unk>',

    '<pad>',

    '<bos>',

    '<eos>'
]

#vocab building

def build_vocab(sentences):

    counter = Counter()

    for sentence in sentences:

        counter.update(sentence)

    word2idx = {}

    idx2word = {}



    for idx, token in enumerate(special_tokens):

        word2idx[token] = idx

        idx2word[idx] = token



    for word in counter.keys():

        if word not in word2idx:

            idx = len(word2idx)

            word2idx[word] = idx

            idx2word[idx] = word

    return word2idx, idx2word



src_word2idx, src_idx2word = build_vocab(

    [x[0] for x in train_dataset]
)

trg_word2idx, trg_idx2word = build_vocab(

    [x[1] for x in train_dataset]
)

#idxes

PAD_IDX = src_word2idx['<pad>']

BOS_IDX = src_word2idx['<bos>']

EOS_IDX = src_word2idx['<eos>']

UNK_IDX = src_word2idx['<unk>']

print("\nEnglish Vocab:", len(src_word2idx))

print("Hindi Vocab:", len(trg_word2idx))


English Vocab: 49859
Hindi Vocab: 44091


In [31]:

def numericalize(tokens, vocab_dict):

    return [

        vocab_dict.get(token, UNK_IDX)

        for token in tokens
    ]

def collate_fn(batch):

    src_batch = []

    trg_batch = []

    for src_sample, trg_sample in batch:

        src_tensor = torch.tensor(

            [BOS_IDX] +

            numericalize(
                src_sample,
                src_word2idx
            ) +

            [EOS_IDX]
        )

        trg_tensor = torch.tensor(

            [BOS_IDX] +

            numericalize(
                trg_sample,
                trg_word2idx
            ) +

            [EOS_IDX]
        )

        src_batch.append(src_tensor)

        trg_batch.append(trg_tensor)

    src_batch = pad_sequence(

        src_batch,

        padding_value=PAD_IDX
    )

    trg_batch = pad_sequence(

        trg_batch,

        padding_value=PAD_IDX
    )

    return src_batch, trg_batch

#loader

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    collate_fn=collate_fn
)

print("\nDataLoaders Ready!")


DataLoaders Ready!


In [32]:
#pos. encoding

class PositionalEncoding(nn.Module):

    def __init__(

        self,

        emb_size,

        dropout,

        maxlen=5000
    ):

        super().__init__()

        den = torch.exp(

            - torch.arange(
                0,
                emb_size,
                2
            ) * math.log(10000) / emb_size
        )

        pos = torch.arange(

            0,

            maxlen
        ).reshape(maxlen, 1)

        pos_embedding = torch.zeros(

            (maxlen, emb_size)
        )

        pos_embedding[:, 0::2] = torch.sin(

            pos * den
        )

        pos_embedding[:, 1::2] = torch.cos(

            pos * den
        )

        pos_embedding = pos_embedding.unsqueeze(-2)

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(

            'pos_embedding',

            pos_embedding
        )

    def forward(self, token_embedding):

        return self.dropout(

            token_embedding +

            self.pos_embedding[
                :token_embedding.size(0), :
            ]
        )

In [33]:
#TRANSFORMER

class Seq2SeqTransformer(nn.Module):

    def __init__(

        self,

        num_encoder_layers,

        num_decoder_layers,

        emb_size,

        nhead,

        src_vocab_size,

        tgt_vocab_size,

        dim_feedforward=512,

        dropout=0.1
    ):

        super().__init__()

        self.transformer = nn.Transformer(

            d_model=emb_size,

            nhead=nhead,

            num_encoder_layers=num_encoder_layers,

            num_decoder_layers=num_decoder_layers,

            dim_feedforward=dim_feedforward,

            dropout=dropout
        )

        self.generator = nn.Linear(

            emb_size,

            tgt_vocab_size
        )

        self.src_tok_emb = nn.Embedding(

            src_vocab_size,

            emb_size
        )

        self.tgt_tok_emb = nn.Embedding(

            tgt_vocab_size,

            emb_size
        )

        self.positional_encoding = PositionalEncoding(

            emb_size,

            dropout
        )

    def forward(

        self,

        src,

        trg,

        src_mask,

        tgt_mask,

        src_padding_mask,

        tgt_padding_mask,

        memory_key_padding_mask
    ):

        src_emb = self.positional_encoding(

            self.src_tok_emb(src)
        )

        tgt_emb = self.positional_encoding(

            self.tgt_tok_emb(trg)
        )

        outs = self.transformer(

            src_emb,

            tgt_emb,

            src_mask,

            tgt_mask,

            None,

            src_padding_mask,

            tgt_padding_mask,

            memory_key_padding_mask
        )

        return self.generator(outs)

In [34]:
#MASKING

def generate_square_subsequent_mask(sz):

    mask = torch.triu(

        torch.ones(
            (sz, sz),
            device=DEVICE
        )
    ) == 1

    mask = mask.transpose(0, 1)

    mask = mask.float().masked_fill(

        mask == 0,

        float('-inf')

    ).masked_fill(

        mask == 1,

        float(0.0)
    )

    return mask

def create_mask(src, tgt):

    src_seq_len = src.shape[0]

    tgt_seq_len = tgt.shape[0]

    tgt_mask = generate_square_subsequent_mask(

        tgt_seq_len
    )

    src_mask = torch.zeros(

        (src_seq_len, src_seq_len),

        device=DEVICE
    ).type(torch.bool)

    src_padding_mask = (

        src == PAD_IDX
    ).transpose(0, 1)

    tgt_padding_mask = (

        tgt == PAD_IDX
    ).transpose(0, 1)

    return (

        src_mask,

        tgt_mask,

        src_padding_mask,

        tgt_padding_mask
    )



SRC_VOCAB_SIZE = len(src_word2idx)

TGT_VOCAB_SIZE = len(trg_word2idx)

model = Seq2SeqTransformer(

    NUM_ENCODER_LAYERS,

    NUM_DECODER_LAYERS,

    EMB_SIZE,

    NHEAD,

    SRC_VOCAB_SIZE,

    TGT_VOCAB_SIZE,

    FFN_HID_DIM,

    DROPOUT
)

model = model.to(DEVICE)

print(model)

# ==========================================
# LOSS FUNCTION
# ==========================================

loss_fn = nn.CrossEntropyLoss(

    ignore_index=PAD_IDX,

    label_smoothing=0.1
)

#optimizer

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LR,

    weight_decay=1e-4
)

Seq2SeqTransformer(
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=256, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): TransformerDecoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerDecoderLayer(
          (self_attn): MultiheadAttent

In [35]:
# TRAINING

best_test_loss = float('inf')

early_stop_counter = 0

best_weights = None

train_losses = []

test_losses = []

print("\n TRAINING STARTED...\n")

for epoch in range(EPOCHS):

    # ======================================
    # TRAIN MODE
    # ======================================

    model.train()

    total_train_loss = 0

    progress_bar = tqdm(

        train_loader,

        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for src, tgt in progress_bar:

        src = src.to(DEVICE)

        tgt = tgt.to(DEVICE)

        tgt_input = tgt[:-1, :]

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(

            src,

            tgt_input
        )

        logits = model(

            src,

            tgt_input,

            src_mask,

            tgt_mask,

            src_padding_mask,

            tgt_padding_mask,

            src_padding_mask
        )

        optimizer.zero_grad()

        tgt_out = tgt[1:, :]

        loss = loss_fn(

            logits.reshape(

                -1,

                logits.shape[-1]
            ),

            tgt_out.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=1.0
        )

        optimizer.step()

        total_train_loss += loss.item()

        progress_bar.set_postfix(

            loss=loss.item()
        )

    avg_train_loss = (

        total_train_loss /

        len(train_loader)
    )

    train_losses.append(avg_train_loss)

    # ======================================
    # TEST EVALUATION
    # ======================================

    model.eval()

    total_test_loss = 0

    with torch.no_grad():

        for src, tgt in test_loader:

            src = src.to(DEVICE)

            tgt = tgt.to(DEVICE)

            tgt_input = tgt[:-1, :]

            src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(

                src,

                tgt_input
            )

            logits = model(

                src,

                tgt_input,

                src_mask,

                tgt_mask,

                src_padding_mask,

                tgt_padding_mask,

                src_padding_mask
            )

            tgt_out = tgt[1:, :]

            loss = loss_fn(

                logits.reshape(

                    -1,

                    logits.shape[-1]
                ),

                tgt_out.reshape(-1)
            )

            total_test_loss += loss.item()

    avg_test_loss = (

        total_test_loss /

        len(test_loader)
    )

    test_losses.append(avg_test_loss)

    # ======================================
    # PRINT METRICS
    # ======================================

    print("\n====================================")

    print(f" Epoch {epoch+1}/{EPOCHS}")

    print(f" Train Loss : {avg_train_loss:.4f}")

    print(f" Test Loss  : {avg_test_loss:.4f}")

    print("====================================\n")

    # ======================================
    # EARLY STOPPING
    # ======================================

    if avg_test_loss < best_test_loss - MIN_DELTA:

        best_test_loss = avg_test_loss

        early_stop_counter = 0

        best_weights = copy.deepcopy(

            model.state_dict()
        )

        with open(

            "models/best_transformer.pkl",

            "wb"
        ) as f:

            pickle.dump(

                model.state_dict(),

                f
            )

        print("Best Model Saved!\n")

    else:

        early_stop_counter += 1

        print(

            f" No Improvement "
            f"({early_stop_counter}/{PATIENCE})"
        )

    if early_stop_counter >= PATIENCE:

        print("\n EARLY STOPPING")

        break

# ==========================================
# RESTORE BEST MODEL
# ==========================================

print("\nRestoring Best Model Weights...")

model.load_state_dict(best_weights)

print("Best Model Restored!")

print("\n TRAINING COMPLETE!")


 TRAINING STARTED...



Epoch 1/25: 100%|██████████| 1875/1875 [02:12<00:00, 14.17it/s, loss=7.15]



 Epoch 1/25
 Train Loss : 7.6937
 Test Loss  : 7.3121

Best Model Saved!



Epoch 2/25: 100%|██████████| 1875/1875 [02:13<00:00, 14.09it/s, loss=6.99]



 Epoch 2/25
 Train Loss : 7.0152
 Test Loss  : 6.9895

Best Model Saved!



...


 Epoch 23/25
 Train Loss : 4.6422
 Test Loss  : 6.0406

 No Improvement (1/3)


Epoch 24/25: 100%|██████████| 1875/1875 [02:12<00:00, 14.12it/s, loss=4.93]



 Epoch 24/25
 Train Loss : 4.5922
 Test Loss  : 6.0343

Best Model Saved!



Epoch 25/25: 100%|██████████| 1875/1875 [02:13<00:00, 14.02it/s, loss=4.82]



 Epoch 25/25
 Train Loss : 4.5449
 Test Loss  : 6.0312

Best Model Saved!


Restoring Best Model Weights...
Best Model Restored!

 TRAINING COMPLETE!


In [36]:


with open(

    "models/best_transformer.pkl",

    "rb"
) as f:

    state_dict = pickle.load(f)

model.load_state_dict(state_dict)

model.eval()

# ==========================================
# GREEDY DECODING
# ==========================================

def greedy_decode(

    model,

    src_sentence,

    max_len=50
):

    model.eval()

    tokens = tokenize_en(src_sentence)

    src = torch.tensor(

        [BOS_IDX] +

        numericalize(
            tokens,
            src_word2idx
        ) +

        [EOS_IDX]

    ).unsqueeze(1).to(DEVICE)

    src_mask = torch.zeros(

        (src.size(0), src.size(0)),

        device=DEVICE
    ).type(torch.bool)

    memory = model.transformer.encoder(

        model.positional_encoding(

            model.src_tok_emb(src)
        ),

        src_mask
    )

    ys = torch.tensor(

        [[BOS_IDX]],

        device=DEVICE
    )

    for _ in range(max_len):

        tgt_mask = generate_square_subsequent_mask(

            ys.size(0)

        ).to(DEVICE)

        out = model.transformer.decoder(

            model.positional_encoding(

                model.tgt_tok_emb(ys)
            ),

            memory,

            tgt_mask
        )

        out = model.generator(out)

      #TEMP. SAMPLING

        temperature = 0.7

        probs = torch.softmax(

            out[-1] / temperature,

            dim=-1
        )

        next_word = torch.multinomial(

            probs,

            num_samples=1

        ).item()

        ys = torch.cat(

            [

                ys,

                torch.tensor(
                    [[next_word]],
                    device=DEVICE
                )
            ],

            dim=0
        )

        if next_word == EOS_IDX:

            break

    translated_tokens = []

    for idx in ys.flatten():

        token = trg_idx2word[idx.item()]

        if token not in [

            '<bos>',

            '<eos>',

            '<pad>'
        ]:

            translated_tokens.append(token)

    return " ".join(translated_tokens)

#TEST SENTENCES

sentences = [

    "How are you",

    "What is your name",

    "I love India",

    "Machine learning is amazing",

    "This is a transformer model"
]

for sentence in sentences:

    translation = greedy_decode(

        model,

        sentence
    )

    print("\n============================")

    print("English :", sentence)

    print("Hindi   :", translation)

    print("============================")


English : How are you
Hindi   : तुम कैसे कैसे हो?

English : What is your name
Hindi   : तुम्हारी नाम है।

English : I love India
Hindi   : मैं ठीक हूं।

English : Machine learning is amazing
Hindi   : मिस अधिक संभावना बराबर है।

English : This is a transformer model
Hindi   : यह बहुत बड़ा है.


In [37]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.3 MB/s eta 0:00:00


In [38]:
import sacrebleu

In [39]:
#BLEU SCORE

def evaluate_bleu(

    model,

    dataset,

    num_samples=100
):

    predictions = []

    references = []

    print("\n Calculating BLEU Score...\n")

    for i in range(num_samples):

       

        src_tokens, trg_tokens = dataset[i]

      

        src_sentence = " ".join(src_tokens)

    

        reference = " ".join(trg_tokens)

       

        prediction = greedy_decode(

            model,

            src_sentence
        )

        predictions.append(prediction)

        references.append([reference])

     

        if i < 5:

            print("\n========================")

            print("English :", src_sentence)

            print("Actual  :", reference)

            print("Predicted:", prediction)

            print("========================")

    #Calculating BLEU

    bleu = sacrebleu.corpus_bleu(

        predictions,

        list(zip(*references))
    )

    print("\nBLEU SCORE:", bleu.score)

    return bleu.score

In [40]:
bleu_score = evaluate_bleu(

    model,

    test_dataset,

    num_samples=200
)


 Calculating BLEU Score...


English : give shots of injections or pills, but he must be alright soon.
Actual  : सुई लगाओ या गोली खिलाओ लेकिन इसे जल्दी से ठीक करो.
Predicted: ये लोग कुछ नफा के लिए जल्दी नहीं हो सका या फिर वह उससे निराश हो सकता है?

English : they said, “o shuaib, we do not understand much of what you say, and we see that you are weak among us. were it not for your tribe, we would have stoned you. you are of no value to us.”
Actual  : और वह लोग कहने लगे ऐ शुएब जो बाते तुम कहते हो उनमें से अक्सर तो हमारी समझ ही में नहीं आयी और इसमें तो शक नहीं कि हम तुम्हें अपने लोगों में बहुत कमज़ोर समझते है और अगर तुम्हारा क़बीला न होता तो हम तुम को (कब का) संगसार कर चुके होते और तुम तो हम पर किसी तरह ग़ालिब नहीं आ सकते
Predicted: और लोगों ने कहा कि ऐ नूह के सिवा हमें (किसी से क्यों न हो कि हम तुम्हें क्या तुम समझते हो कि ऐसा न हो कि तुम लोग तो हम तुम्हारे पास तुम पढ़ते हैं कि तुम लोग ईमान लाने वाले ही नहीं समझते

English : - yeah.
Actual  : - हाँ.
Predicted: - -

English : if evil befal